In [0]:

from pyspark.sql.functions import *
from pyspark.sql.window import Window
employee_data = [
    (1, "Alice", "IT", 100000),
    (2, "Bob", "IT", 120000),
    (3, "Charlie", "IT", 120000),
    (4, "David", "HR", 90000),
    (5, "Eva", "HR", 95000),
    (6, "Frank", "HR", 95000),
    (7, "Grace", "Finance", 110000),
    (8, "Hank", "Finance", 105000),
    (9, "Ivy", "Finance", 110000),
    (10, "Jack", "Finance", 110000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(employee_data, columns)
df.createOrReplaceTempView("employee")

df.write.format("delta").mode("overwrite").saveAsTable("employee_delta")
df.write.format("delta").mode("overwrite").saveAsTable("employee_delta1")
#df.show()

#find the rank of salary per each department
df.withColumn("sal_rank",dense_rank().over(Window.partitionBy("department").orderBy(col("salary").desc()))).filter(col("sal_rank")==1).show()

# find top 1 salary in each department
df.groupBy("department").agg(max("salary").alias("max_salary")).show()

# find 2nd higheset salary in each department




In [0]:
%sql

SELECT department, salary
FROM employee_delta e1
WHERE 1 = (
    SELECT COUNT(DISTINCT e2.salary)
    FROM employee_delta e2
    WHERE e2.department = e1.department
      AND e2.salary > e1.salary
)

In [0]:
# find duplicate salaries in employee_delta table by each department
%sql
select department,salary,count(*) as count as count from employee_delta group by department,salary having count(*)>1
    


In [0]:
%sql
select * from employee_delta

In [0]:
%sql
select *,rank() over(partition by department order By salary desc) as rank from employee_delta

In [0]:
%sql
select *,dense_rank() over(partition by department order By salary desc) as dense_rank from employee_delta

In [0]:
#find highest salary for each department
df = spark.sql("""
select * from (
    select *, row_number() over(partition by department order by salary desc) as row_number
    from employee_delta
) t 
where row_number = 1 
""")
display(df)

In [0]:
#remove duplicates
from pyspark.sql.functions import *
df = spark.sql("""
select * from (
    select *, row_number() over(partition by department order by salary desc) as row_number
    from employee_delta
) t 
where row_number = 1 
""")
display(df)


In [0]:
%sql
select * from employee_delta1

In [0]:
#running total by department
df = spark.sql("""
select *, sum(salary) over(partition by department order by salary desc) as running_total 
from employee_delta
""")
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col

df.withColumn("running_total",sum(col("salary")).over(Window.partitionBy(col("department")).orderBy(col("salary").desc()))).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, row_number
df.withColumn("row_num",row_number().over(Window.partitionBy(col("department")).orderBy(col("salary").desc()))).filter(col("row_num")==1).show()

In [0]:
from pyspark.sql.functions import *

data = [
    (1, ["python", "sql", "java"]),
    (2, ["sql", "java", "oracle"])
]
df=spark.createDataFrame(data,["id","language"])
display(df)
df_explode=df.select("id",explode("language").alias("language"))
df_explode.show()

In [0]:
%sql
-- create table customer(cust_id int,cust_name string,contact integer,location string);
-- create table orders(order_id int,cust_id int,order_date date,order_amount double);
-- create table invoice(invoice_id int,order_id int,delivery_date date);

-- INSERT INTO customer VALUES
-- (1, 'Ravi', 987654321, 'Bangalore'),
-- (2, 'Sam', 987654322, 'Chennai'),
-- (3, 'John', 987654323, 'Hyderabad'),
-- (4, 'Mike', 987654324, 'Mumbai'),
-- (5, 'David', 987654325, 'Delhi'),
-- (6, 'Sara', 987654326, 'Bangalore'),
-- (7, 'Tom', 987654327, 'Chennai'),
-- (8, 'Jenny', 987654328, 'Hyderabad'),
-- (9, 'Chris', 987654329, 'Mumbai'),
-- (10, 'Anu', 987654330, 'Delhi');


-- INSERT INTO orders VALUES
-- (111, 1, '2026-01-02', 2500),
-- (112, 2, '2026-01-03', 7200),
-- (113, 3, '2026-01-03', 1800),
-- (114, 3, '2026-01-05', 9500),
-- (115, 4, '2026-01-05', 4200),
-- (116, 5, '2026-01-06', 6100),
-- (117, 5, '2026-01-07', 3300),
-- (118, 6, '2026-01-08', 8700),
-- (119, 6, '2026-01-08', 2600),
-- (120, 1, '2026-01-09', 5100),

-- (121, 1, '2026-01-10', 7800),
-- (122, 2, '2026-01-10', 2900),
-- (123, 2, '2026-01-11', 6400),
-- (124, 1, '2026-01-12', 1200),
-- (125, 5, '2026-01-12', 9900),
-- (126, 6, '2026-01-13', 4500),
-- (127, 1, '2026-01-13', 7600),
-- (128, 4, '2026-01-14', 3400),
-- (129, 3, '2026-01-14', 8800),
-- (130, 1, '2026-01-15', 1500);

-- INSERT INTO invoice VALUES
-- (1011, 111, '2026-01-04'),
-- (1012, 112, '2026-01-06'),
-- (1013, 113, '2026-01-05'),
-- (1014, 114, '2026-01-09'),
-- (1015, 115, '2026-01-07'),
-- (1016, 116, '2026-01-10'),
-- (1017, 117, '2026-01-11'),
-- (1018, 118, '2026-01-13'),
-- (1019, 119, '2026-01-10'),
-- (1020, 120, '2026-01-14'),

-- (1021, 121, '2026-01-15'),
-- (1022, 122, '2026-01-12'),
-- (1023, 123, '2026-01-16'),
-- (1024, 124, '2026-01-13'),
-- (1025, 125, '2026-01-18'),
-- (1026, 126, '2026-01-17'),
-- (1027, 127, '2026-01-19'),
-- (1028, 128, '2026-01-16'),
-- (1029, 129, '2026-01-20'),
-- (1030, 130, '2026-01-18');

update invoice set delivery_date=null where invoice_id=1011


    


In [0]:
%sql
--generate the 3rd highest location interms of order amount
select location,total_amount from(select c.location location,sum(o.order_amount) as total_amount,dense_rank() over(order by sum(o.order_amount) desc) as rank from customer c join orders o on c.cust_id=o.cust_id group by c.location)t where rank=3


In [0]:
%sql
--find average time gap between order and delivery date
select floor(avg(datediff(i.delivery_date,o.order_date))) as average_delivery_date from orders o inner join invoice i on o.order_id=i.order_id

In [0]:
%sql
-- identify customer details who has not received their order
select c.cust_id,c.cust_name,c.location,c.contact from customer c join orders o on c.cust_id=o.cust_id join invoice i on o.order_id=i.order_id where i.delivery_date is null

In [0]:
%sql
-- count orders for each customer
select c.cust_name,count(o.order_id) as total_orders from customer c join orders o on c.cust_id=o.cust_id group by c.cust_name

In [0]:
%sql
--Average order amount per customer
select c.cust_name,floor(avg(o.order_amount)) from customer c join orders o on c.cust_id=o.cust_id group by c.cust_name
    
